# 50. 小提琴图（violinplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 7 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 箱线图（boxplot）  →  **本章任务：** 小提琴图（violinplot）  →  **下一步：** 抖动散点图（stripplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

平均值只能告诉你数据的“中心”，很多信息其实藏在分布的形状里——
有的品类客单价很集中，有的却拖着一长条高消费的尾巴。



## 本章目标

学完本章，你将能够：

- **理解**：理解「小提琴图（violinplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「小提琴图（violinplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「小提琴图（violinplot）」并读出其中的结论。


## 50.1 适用场景

**背景引入**：平均值只能告诉你数据的“中心”，很多信息其实藏在分布的形状里——
有的品类客单价很集中，有的却拖着一长条高消费的尾巴。小提琴图把每个分组的密度曲线画成对称的“提琴”，
既能看出中心位置，也能直观对比各组是偏态还是有多峰。


样本量足够，希望比较多峰、偏态或尾部形状。


## 50.2 数据结构

每组包含较多连续数值观察。


## 50.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 inner="quart" 改为 inner="box"，对比四分位线与箱线摘要的显示效果
2. 调整 bw_adjust 参数（如 0.5 或 2.0），说明带宽对密度曲线平滑度的影响
3. 修改 cut 参数从 0 为 2，观察密度曲线在数据范围外的延伸变化


## 50.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.violinplot()`、`ax.set()`、`fig.tight_layout()` | 样本量足够，希望比较多峰、偏态或尾部形状。 | 小样本产生误导性平滑形状 |
| 进阶变体 | `plt.subplots()`、`sns.violinplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 不说明每组样本量 |
| 关键参数 | `inner` | 内部摘要 | 小样本产生误导性平滑形状 |
| 关键参数 | `cut` | 密度延伸 | 不说明每组样本量 |
| 关键参数 | `bw_adjust` | 带宽 | 不同组独立归一化却比较绝对宽度 |
| 关键参数 | `split` | 二分类对称拆分 | 小样本产生误导性平滑形状 |


## 50.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-50 -->
### 数学推导｜核密度估计把样本平滑成分布

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每个样本放置一个核。** 以 $x_i$ 为中心、带宽为 $h$ 的核为

$$
K_h(x-x_i)=\frac{1}{h}K\!\left(\frac{x-x_i}{h}\right)
$$

$1/h$ 保证拉宽曲线后面积仍为 1。

**第 2 步｜把所有小曲线平均。** $n$ 个单位面积核相加后再除以 $n$，总面积仍为 1，于是得到 $\hat f_h(x)$。

**第 3 步｜理解带宽。** 较小 $h$ 保留局部起伏但方差大；较大 $h$ 更平滑但可能抹掉真实结构。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_h(x)=\frac{1}{nh}\sum_{i=1}^{n}K\!\left(\frac{x-x_i}{h}\right)
$$

**符号解释：** $K$ 是核函数，$h$ 是带宽；$h$ 越大曲线越平滑。

**代码对应：** 调整 `bw_adjust`（或旧版带宽参数）并与原始样本/直方图交叉检查。

**使用边界：** KDE 会在观测范围外延伸，小样本或有自然边界的数据尤其要谨慎。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 50.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.violinplot(
    data=orders,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    inner="quart",
    cut=0,
    ax=ax,
)
ax.set(title="品类客单价密度", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


**练一练**：基础小提琴图只改了一个参数，试试就能体会它的作用。请在下一个单元格中，把 `sns.violinplot` 的 `inner="quart"` 改成 `inner="box"`，运行后再对比：四分位线与箱线摘要显示上有什么差别，哪一个更像箱线图？提示：横轴用 `x="category"`、纵轴用 `y="order_value"`、数据来自 `orders`；画完后用 `orders.groupby("category").size()` 统计每个类别的记录数完成自检。


In [ ]:
# 请在下方填写代码


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.6))
sns.violinplot(
    data=orders,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    inner="box",
    cut=0,
    ax=ax,
)
ax.set(title="品类客单价（箱线摘要）", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 50.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

two_channels = orders[orders["channel"].isin(["自然流量", "广告"])]
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.violinplot(
    data=two_channels,
    x="category",
    y="order_value",
    hue="channel",
    split=True,
    inner="quart",
    cut=0,
    palette=["#1a73e8", "#f9ab00"],
    ax=ax,
)
ax.set(title="自然流量与广告客单价密度", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 50.8 参数说明

- inner：内部摘要
- cut：密度延伸
- bw_adjust：带宽
- split：二分类对称拆分


## 50.9 结果解读

宽处表示估计密度较高；形状受带宽影响，不等于实际频数。


## 50.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 50.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 50.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 50.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 50.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 50.12 易错点提醒

- 小样本产生误导性平滑形状
- 不说明每组样本量
- 不同组独立归一化却比较绝对宽度


## 50.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 50.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：用 split 参数把渠道拆在同一把琴上，观察对比
# 【目标】split=True 让左右半琴各表一个渠道，直接对比两组密度。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：筛选两个渠道，用 split=True 左右对比。
two_channels = orders[orders["channel"].isin(["自然流量", "广告"])]
fig, ax = plt.subplots(figsize=(8, 4.6))
sns.violinplot(
    data=two_channels,
    x="category",
    y="order_value",
    hue="channel",
    split=True,
    inner="quart",
    cut=0,
    palette=["#1a73e8", "#f9ab00"],
    ax=ax,
)
ax.set(title="两渠道客单价密度对比", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：左右半琴对比，两组密度有何不同 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(
    data=orders,
    x="region",
    y="items",
    hue="region",
    legend=False,
    inner="box",
    cut=0,
    palette="pastel",
    ax=ax,
)
ax.set(title="区域购买件数分布", xlabel="区域", ylabel="件数")
fig.tight_layout()
plt.show()


## 50.15 小结

用小提琴图展示分类组的平滑密度形状和中心信息。


### 50.15.1 你已经掌握

- 判断小提琴图（violinplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 50.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `inner` | 内部摘要 |
| `cut` | 密度延伸 |
| `bw_adjust` | 带宽 |
| `split` | 二分类对称拆分 |


### 50.15.3 需要注意

- 小样本产生误导性平滑形状
- 不说明每组样本量
- 不同组独立归一化却比较绝对宽度


### 50.15.4 完成检查

- [ ] 能判断什么问题适合使用小提琴图（violinplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 50.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
